In [ ]:
# 1. FIX THE WARNING: Unsloth must be imported first!
import unsloth
import os
import torch
from training_hub import lora_sft

# 2. CONFIGURATION
MODEL_NAME = "mistralai/Mistral-Small-24B-Instruct-2501" 
DATA_PATH = "sarhachat_training_data.jsonl"
OUTPUT_DIR = "./sarhachat-lora-adapter"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🚀 Launching Red Hat Training Hub for {MODEL_NAME}...")

try:
    # 3. RUN THE WRAPPER (This does all the heavy lifting)
    result = lora_sft(
        model_path=MODEL_NAME,
        data_path=DATA_PATH,
        ckpt_output_dir=OUTPUT_DIR,
        
        # LoRA config
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        
        # 4-Bit Quantization (REQUIRED so Mistral 24B fits on your L40S)
        load_in_4bit=True,
        
        # Training Settings
        num_epochs=2,
        learning_rate=2e-4,
        max_seq_len=2048,
        micro_batch_size=2,
        gradient_accumulation_steps=4, # Simulates a batch size of 8
        
        # Optimizations
        bf16=True,
        sample_packing=True,
        
        # Dataset mapping (Mistral has this template natively)
        dataset_type="chat_template",
        field_messages="messages",
        
        logging_steps=2,
        save_steps=100,
    )
    
    print("\n" + "=" * 60)
    print("✅ MISTRAL TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 60)

except Exception as e:
    print(f"\n❌ ERROR: {e}")



In [ ]:

import torch

# (Assuming model and tokenizer are already loaded from your previous cell!)

UNIVERSAL_PERSONA = (
    "You are SARHAchat, a warm, empathetic clinical assistant speaking directly to a patient. "
    "You are in the middle of an ongoing conversation. DO NOT introduce yourself. "
    "DO NOT say 'Hello' or 'I am SARHAchat'. Speak in the first person ('I'). "
    "DO NOT output meta-commentary, suggestions, or scripts to the developer. "
    "Output ONLY the exact message the patient will read. "
    "CRITICAL CONVERSATION RULE: If the user asks a question, expresses fear, or makes a comment, "
    "you MUST briefly and empathetically address it in ONE sentence before asking your next required question."
)

test_prompts = [
    "I'm really scared about getting an IUD, does it hurt? I don't know if I can do it.",
    "Why are you asking me all these questions? Just tell me what birth control to take so I can leave."
]

print("\n" + "█"*75)
print("🚀 STARTING A/B TEST: BASE MISTRAL vs. FINE-TUNED SARHAchat")
print("█"*75)

for i, prompt in enumerate(test_prompts):
    print(f"\n🗣️ PATIENT SAYS: \"{prompt}\"")
    print("="*75)
    
    messages = [
        {"role": "system", "content": UNIVERSAL_PERSONA},
        {"role": "user", "content": prompt}
    ]
    
    # FIX: We explicitly ask for a dictionary so it includes the attention_mask!
    inputs = tokenizer.apply_chat_template(
        messages, 
        tokenize=True, 
        add_generation_prompt=True, 
        return_tensors="pt",
        return_dict=True  # <--- THIS IS THE MAGIC FIX
    ).to("cuda")
    
    # ---------------------------------------------------------
    # TEST 1: THE RAW BASE MODEL (ADAPTER OFF)
    # ---------------------------------------------------------
    with model.disable_adapter():
        # FIX: We unpack the dictionary using **inputs
        outputs_base = model.generate(**inputs, max_new_tokens=150, temperature=0.3, use_cache=True)
        response_base = tokenizer.decode(outputs_base[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        
        print("\n🛑 BASE MISTRAL 24B (Generic AI):")
        print(response_base.strip())
        print("-" * 75)
    
    # ---------------------------------------------------------
    # TEST 2: YOUR FINE-TUNED MODEL (ADAPTER ON)
    # ---------------------------------------------------------
    outputs_lora = model.generate(**inputs, max_new_tokens=150, temperature=0.3, use_cache=True)
    response_lora = tokenizer.decode(outputs_lora[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    print("\n✨ SARHAchat (Your Fine-Tuned Model):")
    print(response_lora.strip())
    print("\n" + "█"*75)


In [ ]:
import unsloth
import os
import torch
from training_hub import lora_sft

MODEL_NAME = "mistralai/Mistral-Small-24B-Instruct-2501" 
DATA_PATH = "sarhachat_training_data.jsonl"
OUTPUT_DIR = "./sarhachat-lora-vA-10epochs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"🚀 Launching Version A: High Repetition...")

lora_sft(
    model_path=MODEL_NAME, data_path=DATA_PATH, ckpt_output_dir=OUTPUT_DIR,
    lora_r=16, lora_alpha=32, lora_dropout=0.0, load_in_4bit=True,
    num_epochs=10,  # <--- CHANGED: Increased from 2 to 10
    learning_rate=2e-4, max_seq_len=2048, micro_batch_size=2, gradient_accumulation_steps=4,
    bf16=True, sample_packing=True, dataset_type="chat_template", field_messages="messages",
    logging_steps=2, save_steps=100,
)
print(f"✅ Version A saved to {OUTPUT_DIR}")




In [ ]:
import unsloth
import os
import torch
from training_hub import lora_sft

MODEL_NAME = "mistralai/Mistral-Small-24B-Instruct-2501" 
DATA_PATH = "sarhachat_training_data.jsonl"
OUTPUT_DIR = "./sarhachat-lora-vB-rank64"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"🚀 Launching Version B: High Capacity...")

lora_sft(
    model_path=MODEL_NAME, data_path=DATA_PATH, ckpt_output_dir=OUTPUT_DIR,
    lora_r=64,      # <--- CHANGED: Increased from 16 to 64
    lora_alpha=128, # <--- CHANGED: Scaled to match the new rank
    lora_dropout=0.0, load_in_4bit=True,
    num_epochs=2,   # <--- Back to 2 epochs
    learning_rate=2e-4, max_seq_len=2048, micro_batch_size=2, gradient_accumulation_steps=4,
    bf16=True, sample_packing=True, dataset_type="chat_template", field_messages="messages",
    logging_steps=2, save_steps=100,
)
print(f"✅ Version B saved to {OUTPUT_DIR}")



In [ ]:
import unsloth
import os
import torch
from training_hub import lora_sft

MODEL_NAME = "mistralai/Mistral-Small-24B-Instruct-2501" 
DATA_PATH = "sarhachat_training_data.jsonl"
OUTPUT_DIR = "./sarhachat-lora-vC-sledgehammer"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"🚀 Launching Version C: Sledgehammer...")

lora_sft(
    model_path=MODEL_NAME, data_path=DATA_PATH, ckpt_output_dir=OUTPUT_DIR,
    lora_r=64,      # <--- CHANGED: High Capacity
    lora_alpha=128, 
    lora_dropout=0.0, load_in_4bit=True,
    num_epochs=10,  # <--- CHANGED: High Repetition
    learning_rate=2e-4, max_seq_len=2048, micro_batch_size=2, gradient_accumulation_steps=4,
    bf16=True, sample_packing=True, dataset_type="chat_template", field_messages="messages",
    logging_steps=2, save_steps=100,
)
print(f"✅ Version C saved to {OUTPUT_DIR}")





In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import os

ADAPTER_DIR = "./sarhachat-lora-vA-10epochs"
FUSED_OUTPUT_DIR = "./sarhachat-vA-fused"
os.makedirs(FUSED_OUTPUT_DIR, exist_ok=True)

print(f"⚙️ Loading Base Model for fusing...")
# Load the base model in full precision
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-Small-24B-Instruct-2501")
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-Small-24B-Instruct-2501",
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    low_cpu_mem_usage=True
)

print(f"⚙️ Merging {ADAPTER_DIR} into base model...")
# Attach adapter and fuse it permanently!
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model = model.merge_and_unload()

print(f"💾 Saving standalone model to {FUSED_OUTPUT_DIR}...")
model.save_pretrained(FUSED_OUTPUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(FUSED_OUTPUT_DIR)

print("✅ FUSING COMPLETE! You now have a standalone AI model.")



